<a href="https://colab.research.google.com/github/Kitrainsom/meridian/blob/main/AS_Meridian_regroupingv_v3_ms_ads_out%2Bimps_linkedin_reddit_x.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# **Introduction to Meridian Demo**

Welcome to the Meridian end-to-end demo. This simplified demo showcases the fundamental functionalities and basic usage of the library, including working examples of the major modeling steps:


<ol start="0">
  <li><a href="#install">Install</a></li>
  <li><a href="#load-data">Load the data</a></li>
  <li><a href="#configure-model">Configure the model</a></li>
  <li><a href="#model-diagnostics">Run model diagnostics</a></li>
  <li><a href="#generate-summary">Generate model results & two-page output</a></li>
  <li><a href="#generate-optimize">Run budget optimization & two-page output</a></li>
  <li><a href="#save-model">Save the model object</a></li>
</ol>


Note that this notebook skips all of the exploratory data analysis and preprocessing steps. It assumes that you have completed these tasks before reaching this point in the demo.

This notebook utilizes sample data. As a result, the numbers and results obtained might not accurately reflect what you encounter when working with a real dataset.

<a name="install"></a>
## Step 0: Install

1\. Make sure you are using one of the available GPU Colab runtimes which is **required** to run Meridian. You can change your notebook's runtime in `Runtime > Change runtime type` in the menu. All users can use the T4 GPU runtime which is sufficient to run the demo colab, free of charge. Users who have purchased one of Colab's paid plans have access to premium GPUs (such as V100, A100 or L4 Nvidia GPU).

2\. Install the latest version of Meridian, and verify that GPU is available.

In [ ]:
# Install meridian: from PyPI @ latest release
!pip install --upgrade google-meridian[colab,and-cuda]

# Install meridian: from PyPI @ specific version
# !pip install google-meridian[colab,and-cuda]==1.0.3

# Install meridian: from GitHub @HEAD
# !pip install --upgrade "google-meridian[colab,and-cuda] @ git+https://github.com/google/meridian.git"

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az

import IPython

from meridian import constants
from meridian.data import load
from meridian.data import test_utils
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import optimizer
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import summarizer
from meridian.analysis import formatter


# check if GPU is available
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Num CPUs Available: ", len(tf.config.experimental.list_physical_devices('CPU')))

<a name="load-data"></a>
## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_all_channels.csv) as follows.

1\. Map the column names to their corresponding variable types. For example, the column names 'GQV' and 'Competitor_Sales' are mapped to `controls`. The required variable types are `time`, `controls`, `population`, `kpi`, `revenue_per_kpi`, `media` and `spend`. If your data includes organic media or non-media treatments, you can add them using `organic_media` and `non_media_treatments` arguments. For the definition of each variable, see
[Collect and organize your data](https://developers.google.com/meridian/docs/user-guide/collect-data).

In [ ]:


df = pd.read_csv("/content/data4mmm2y_imps.csv", parse_dates=["time"])
df["time"] = pd.to_datetime(df["time"])
df.set_index("time", inplace=True)

weekly = df.resample('W-MON', closed='left', label='left').agg({
    'holidays': 'max',
    'Ramadan': 'max',
    'population': 'first',
    # budgets → sum
    'budget_Apple Search Ads': 'sum',
    'budget_Display': 'sum',
    'budget_Facebook': 'sum',
    'budget_G Ads': 'sum',
    'budget_LinkedIn': 'sum',
    'budget_MS Ads': 'sum',
    #'budget_Native Ads': 'sum',
    'budget_Outbrain': 'sum',
    'budget_Reddit': 'sum',
    'budget_Taboola': 'sum',
    'budget_Twitter': 'sum',
    'budget_OOH': 'sum',
    'budget_total' : 'sum',


       # metrics
    'impressions_G Ads': 'sum',
    'impressions_Facebook': 'sum',
    'impressions_Apple Search Ads': 'sum',
    'impressions_MS Ads': 'sum',
    'impressions_OOH': 'sum',
    'impressions_Display': 'sum',
    'impressions_LinkedIn': 'sum',
    'impressions_Outbrain': 'sum',
     'impressions_Reddit': 'sum',
    'impressions_Taboola': 'sum',
    'impressions_Twitter': 'sum',
    'leads': 'sum',
    'applications_created': 'sum',
    'applications_submitted': 'sum',
    'applications_approved': 'sum',
    'accounts_created': 'sum',
    'first_traded_accounts': 'sum',
    # trends / indices → mean or last
    'google_trends_stock_interest': 'mean',
    'brand_search_index': 'mean',
    # flags
    'is_weekend': 'max',
    'high_vix': 'mean',
    'strong_usd': 'mean',
    'Dubai_Crude_USD_smooth':'mean',
    'UAE_GDP_Growth_Q_%' : 'mean',
    # sessions etc.
    'sessions': 'sum',
    'engaged_sessions': 'sum',
    'organic_sessions': 'sum',


    # ... add all other columns with sensible agg
}).reset_index()

weekly.to_csv("weekly_data.csv", index=False)

# Channels Grouping


In [ ]:
df = pd.read_csv("/content/weekly_data.csv", parse_dates=["time"])


# Define the final grouping strategy
groups = {

    'impressions_The_Other': ['impressions_Apple Search Ads','impressions_Twitter','impressions_LinkedIn', 'impressions_Reddit', 'impressions_Outbrain', 'impressions_Taboola'],
    'budget_The_Other':['budget_Apple Search Ads','budget_Twitter','budget_LinkedIn', 'budget_Reddit', 'budget_Outbrain', 'budget_Taboola']
}

# Aggregate the spend in your dataframe
for group_name, sub_channels in groups.items():
    # .sum(axis=1) combines the daily spend of all sub-channels
    df[group_name] = df[sub_channels].sum(axis=1)

# Save the new aggregated file
df.to_csv('/content/data_groups_channels_weekly.csv', index=False)
print("File saved with +1 group")

In [ ]:
df.shape



# Correlation between Spend and Impressions


In [ ]:
import pandas as pd

spend_cols = [
    'budget_G Ads', 'budget_OOH', 'budget_Facebook', 'budget_Display',
    'budget_MS Ads', 'budget_Twitter', 'budget_LinkedIn', 'budget_Reddit',
    'budget_The_Other'
]

# 1) Correlation matrix between all spend columns
corr_spend_vs_spend = df[spend_cols].corr()

print("1) Correlation Matrix: Spend vs Spend")
print(corr_spend_vs_spend)
print("-" * 50)

# 2) Correlations between all spend columns and 'applications_submitted'
corr_spend_vs_apps = df[spend_cols].corrwith(df['applications_submitted'])

print("2) Correlation: Spend vs Applications Submitted")
print(corr_spend_vs_apps)
print("-" * 50)

# 3) Correlations between all spend columns and 'leads'
corr_spend_vs_leads = df[spend_cols].corrwith(df['leads'])

print("3) Correlation: Spend vs Leads")
print(corr_spend_vs_leads)

#Budget share


In [ ]:
#Primary channels

import pandas as pd

# 1. Define your main media columns
core_budget_cols = [
    'budget_G Ads',
    'budget_Facebook',
    'budget_Display',
    'budget_MS Ads',
    'budget_The_Other',
    'budget_OOH'
]

# 2. Calculate the "Other" spend
# We assume 'budget_total' is your column for the total weekly/daily spend
total_wallet = df['budget_total'].sum()
core_spend_sum = df[core_budget_cols].sum().sum()
other_spend = total_wallet - core_spend_sum

# 3. Create a summary dataframe
summary_data = df[core_budget_cols].sum().reset_index()
summary_data.columns = ['Channel', 'Total_Spend']

# 4. Append the 'Other' row
other_row = pd.DataFrame([{'Channel': 'other', 'Total_Spend': other_spend}])
budget_df = pd.concat([summary_data, other_row], ignore_index=True)

# 5. Calculate Share %
budget_df['Spend_Share'] = budget_df['Total_Spend'] / total_wallet

# 6. Format for the Slide
print_df = budget_df.copy()
print_df['Spend_Share'] = print_df['Spend_Share'].map('{:.2%}'.format)
print_df['Total_Spend'] = print_df['Total_Spend'].map('{:,.0f}'.format)

print(f"--- Total Marketing Wallet: {total_wallet:,.0f} ---")
print(print_df)

# CONFIGURATION OF INPUTS


In [ ]:
df = pd.read_csv('/content/data_groups_channels_weekly.csv', parse_dates=["time"])
# 1. Define the Channel Names (Labels for charts)
# Order: G_Search, Facebook, Display, MS_ads, Other, OOH
channel_names = [
    'G_Search',
    'Facebook',
    'Display_Video',
    'MS Ads',
    'The_Other',
    'OOH'
]

# 2. Define the Signal Columns (Inputs for impact)
# NOTE: These must match the column names in your new dataframe
media_signal_cols = [
    'impressions_G Ads',      #  Impressions
    'impressions_Facebook',      #  Impressions
    'impressions_Display',      # Impressions
    'impressions_MS Ads', # Aggregated Impressions
    'impressions_The_Other',           # Aggregated Impressions
    'impressions_OOH'                # The 0-1 Scaled Proxy
]

# 3. Define the Cost Columns
# NOTE: These are the 'budget_' versions of the same groups
media_cost_cols = [
    'budget_G Ads',
    'budget_Facebook',
    'budget_Display',
    'budget_MS Ads',
    'budget_The_Other',
    'budget_OOH'                      # Use actual spend for ROI
]

# 4. Create Mappings (Optional, mostly for checking alignment)
correct_media_to_channel = dict(zip(media_signal_cols, channel_names))
correct_media_spend_to_channel = dict(zip(media_cost_cols, channel_names))

# 5. Initialize the CoordToColumns Object
coord_to_columns = load.CoordToColumns(
    time='time',
    controls=[
        'holidays',
        'Ramadan',
        #'google_trends_stock_interest',
        'high_vix',
        'strong_usd',
        'Dubai_Crude_USD_smooth',
        'UAE_GDP_Growth_Q_%'
    ],
    kpi='applications_submitted',
    population='population',
    media=media_signal_cols,
    media_spend=media_cost_cols,
    organic_media=['organic_sessions'] # Good to keep this separate
)

print("CoordToColumns configuration updated for 6-Group Strategy.")

3\. Load the CSV data using `CsvDataLoader`. Note that `csv_path` is the path to the data file location.

In [ ]:
loader = load.CsvDataLoader(
    csv_path="/content/data_groups_channels_weekly.csv",
    kpi_type='non_revenue',
    coord_to_columns=coord_to_columns,
    media_to_channel=correct_media_to_channel,
    media_spend_to_channel=correct_media_spend_to_channel
)
data = loader.load()

Note that the simulated data here does not contain reach and frequency. We recommend including reach and frequency data whenever they are available. For information about the advantages of utilizing reach and frequency, see [Bayesian Hierarchical Media Mix Model Incorporating Reach and Frequency Data](https://research.google/pubs/bayesian-hierarchical-media-mix-model-incorporating-reach-and-frequency-data/#:~:text=By%20incorporating%20R%26F%20into%20MMM,based%20on%20optimal%20frequency%20recommendations.). For code snippet for loading reach and frequency data, see [Load geo-level data with reach and frequency](https://developers.google.com/meridian/docs/user-guide/load-geo-data-with-rf)

The documentation provides guidance for instances where reach and frequency data is accessible for specific channels. Additionally, for information about how to load other data types and formats, including data with reach and frequency, see [Supported data types and formats](https://developers.google.com/meridian/docs/user-guide/supported-data-types-formats).

<a name="configure-model"></a>
## Step 2: Configure the model

Meridian uses Bayesian framework and Markov Chain Monte Carlo (MCMC) algorithms to sample from the posterior distribution.

1\. Inititalize the `Meridian` class by passing the loaded data and the customized model specification. One advantage of Meridian lies in its capacity to calibrate the model directly through ROI priors, as described in [Media Mix Model Calibration With Bayesian Priors](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/). In this particular example, the ROI priors for all media channels are identical, with each being represented as Lognormal(0.2, 0.9).

In [ ]:
#24Feb - application ms ads tight -  hope final

import tensorflow_probability as tfp
from meridian.model import spec, prior_distribution

# --- 1. PAID MEDIA PRIORS ---
# Order: [G_Search, Facebook, Display, MS Ads, Other, OOH]
target_shares = [0.39, 0.09, 0.06, 0.05, 0.03, 0.10]

# THE FIX: Increased the 4th value (MS Ads) from 20.0 to 200.0
strength_values = [50.0, 30.0, 30.0, 200.0, 20.0, 100.0]

c1_m = [t * s for t, s in zip(target_shares, strength_values)]
c0_m = [s - (t * s) for t, s in zip(target_shares, strength_values)]

# Adstock Priors
alpha_c1 = [4.6, 3.0, 5.0, 1.0, 3.0, 7.9]
alpha_c0 = [5.4, 5.0, 4.0, 4.0, 5.0, 2.1]

# --- 2. ORGANIC MEDIA PRIOR (SEO @ 10%) ---
# We use a Beta distribution to strictly enforce the 0 to 1 percentage constraint.
seo_target = 0.10
seo_strength = 300.0
seo_c1 = seo_target * seo_strength # 30.0
seo_c0 = seo_strength - seo_c1     # 270.0

seo_prior = tfp.distributions.Beta(concentration1=seo_c1, concentration0=seo_c0)

# --- 3. ASSEMBLE SPEC ---
model_spec_updated = spec.ModelSpec(
    prior=prior_distribution.PriorDistribution(
        # Paid Media
        contribution_m=tfp.distributions.Beta(concentration1=c1_m, concentration0=c0_m),
        alpha_m=tfp.distributions.Beta(concentration1=alpha_c1, concentration0=alpha_c0),

        # Organic Media
        contribution_om=seo_prior
    ),
    media_prior_type='contribution',
    knots=4,
    max_lag=12,
    hill_before_adstock=False
)

# Initialize the model
mmm = model.Meridian(input_data=data, model_spec=model_spec_updated)

In [ ]:
#24Feb longer adstock for G_Search 3w and OOH 10w, SEO 10% prior

import tensorflow_probability as tfp
from meridian.model import spec, prior_distribution

# --- 1. PAID MEDIA PRIORS ---
target_shares = [0.39, 0.09, 0.06, 0.05, 0.03, 0.10]
strength_values = [50.0, 30.0, 30.0, 20.0, 20.0, 100.0]

c1_m = [t * s for t, s in zip(target_shares, strength_values)]
c0_m = [s - (t * s) for t, s in zip(target_shares, strength_values)]

alpha_c1 = [4.6, 3.0, 5.0, 1.0, 3.0, 7.9]
alpha_c0 = [5.4, 5.0, 4.0, 4.0, 5.0, 2.1]

# --- 2. ORGANIC MEDIA PRIOR (SEO @ 10%) ---
# We use a Beta distribution to strictly enforce the 0 to 1 percentage constraint.
seo_target = 0.1
seo_strength = 300
seo_c1 = seo_target * seo_strength # 30.0
seo_c0 = seo_strength - seo_c1     # 270.0

seo_prior = tfp.distributions.Beta(concentration1=seo_c1, concentration0=seo_c0)

# --- 3. ASSEMBLE SPEC ---
model_spec_updated = spec.ModelSpec(
    prior=prior_distribution.PriorDistribution(
        # Paid Media
        contribution_m=tfp.distributions.Beta(concentration1=c1_m, concentration0=c0_m),
        alpha_m=tfp.distributions.Beta(concentration1=alpha_c1, concentration0=alpha_c0),

        # THE DEFINITIVE FIX: 'contribution_om' (Contribution for Organic Media)
        contribution_om=seo_prior
    ),
    media_prior_type='contribution',
    # organic_media_prior_type defaults to 'contribution', so we don't need to specify it
    knots=4,
    max_lag=12,
    hill_before_adstock=False
)

# Initialize the model
mmm = model.Meridian(input_data=data, model_spec=model_spec_updated)

In [ ]:
#UPDATED 5 Feb'26 MODEL WITH 6-GROUP PRIORS --- knots=3

# Order  match channel_names:
# [G_Search, Facebook, Display_Video, Perf_Search, The_Other, OOH]

# 1. Target Shares (What % of accounts we think they drive)
# Totaling ~0.78 (leaving ~22% for baseline/organic)
target_shares = [0.39, 0.09, 0.06, 0.05, 0.03, 0.10]

# 2. Strength (Confidence in our guess)
# High confidence for Search and OOH; Moderate for the rest
strength_values = [50.0, 30.0, 30.0, 20.0, 20.0, 50.0]

c1_m = [t * s for t, s in zip(target_shares, strength_values)]
c0_m = [s - (t * s) for t, s in zip(target_shares, strength_values)]

# 3. Alpha Priors (Memory/Adstock)
# [Search, FB, Display, Perf_Search, Tail, OOH]
# OOH (Index 5) gets the strongest memory (concentration1=6.0)
alpha_c1 = [1.0, 3.0, 5.0, 1.0, 3.0, 6.0]
alpha_c0 = [4.0, 5.0, 4.0, 4.0, 5.0, 4.0]

model_spec = spec.ModelSpec(
    prior=prior_distribution.PriorDistribution(
        contribution_m=tfp.distributions.Beta(concentration1=c1_m, concentration0=c0_m),
        alpha_m=tfp.distributions.Beta(concentration1=alpha_c1, concentration0=alpha_c0)
    ),
    media_prior_type='contribution',
    knots=3,             #
    max_lag=10,          # Allows OOH "Proxy" signal to have a long tail
    hill_before_adstock=False
)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

2\. Use the `sample_prior()` and `sample_posterior()` methods to obtain samples from the prior and posterior distributions of model parameters. If you are using the T4 GPU runtime this step may take about 10 minutes for the provided data set.

# RUNRUNRUN

In [ ]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(n_chains=8, n_adapt=500, n_burnin=500, n_keep=1000, seed=33)

For more information about configuring the parameters and using a customized model specification, such as setting different ROI priors for each media channel, see [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model).

<a name="model-diagnostics"></a>
## Step 3: Run model diagnostics

After the model is built, you must assess convergence, debug the model if needed, and then assess the model fit.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 4: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)
print(mmm_summarizer)

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.predictive_accuracy_table()

In [ ]:
import pandas as pd
import arviz as az

# 1. Get Media Names
media_names = mmm.inference_data.posterior.coords['media_channel'].values

# 2. Extract Posterior Samples for Media Contribution
# shape: (chains, draws, media)
post_contribs = mmm.inference_data.posterior['contribution_m']

# 3. Calculate Mean and HDI using ArviZ
# az.hdi returns a dataset with 'lower' and 'upper' bounds
hdi_data = az.hdi(post_contribs, hdi_prob=0.94)['contribution_m'].values
means = post_contribs.mean(axis=(0, 1)).values

# 4. Build the Media DataFrame
df_media = pd.DataFrame({
    'Channel': media_names,
    'Contribution %': means * 100,
    'HDI 3%': hdi_data[:, 0] * 100,
    'HDI 97%': hdi_data[:, 1] * 100
})

# 5. Handle Baseline (Calculated as the remainder)
total_media_pct = means.sum()
baseline_pct = 1.0 - total_media_pct
# Note: Baseline HDI is harder to calculate directly this way,
# but usually, we report the media uncertainty specifically.
df_baseline = pd.DataFrame({
    'Channel': ['Baseline (Economy/Organic)'],
    'Contribution %': [baseline_pct * 100],
    'HDI 3%': [None],
    'HDI 97%': [None]
})

final_table = pd.concat([df_media, df_baseline], ignore_index=True).sort_values(by='Contribution %', ascending=False)

print("--- FINAL CONTRIBUTION TABLE WITH UNCERTAINTY ---")
print(final_table.to_string(index=False, float_format=lambda x: f"{x:.2f}%"))

In [ ]:
import arviz as az

# 1. DIAGNOSTIC: Run this to see all available names if the error persists
# print(mmm_ground_truth.inference_data.posterior.data_vars.keys())

# 2. Extract Organic Contribution and Adstock using correct naming (_om)
try:
    organic_results = az.summary(
        mmm.inference_data,
        var_names=['contribution_om', 'alpha_om']
    )

    # Map to your specific channel name
    organic_results.index = ['Organic_Sessions_Contribution', 'Organic_Sessions_Alpha']

    print("--- ORGANIC SESSIONS: EXPLORATION ---")
    print(organic_results[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])

except KeyError:
    print("Variable names might differ. Checking 'beta_om' or 'roi_om'...")
    # Alternate check if the above still fails
    print(mmm.inference_data.posterior.data_vars.keys())

In [ ]:
import arviz as az
import pandas as pd
import numpy as np

# 1. Define your channel names in order
#G_Search, Facebook, Display, Perf_Search, Other, OOH
channels = ['G_Search', 'Facebook', 'Display', 'MS Ads',  'Other', 'OOH']

# 2. Extract Posterior Summary for Alpha
# This gets Mean, HDI (Credible Intervals), and R-hat
alpha_summary = az.summary(mmm.inference_data, var_names=['alpha_m'])

# 3. Extract Prior Mean for Alpha (to see how much the model shifted)
# We take the mean of the prior distribution samples
prior_alpha_mean = mmm.inference_data.prior['alpha_m'].mean(dim=['chain', 'draw']).values

# 4. Build the final DataFrame
df_alphas = pd.DataFrame(index=channels)
df_alphas['Prior_Alpha'] = prior_alpha_mean
df_alphas['Posterior_Alpha'] = alpha_summary['mean'].values
df_alphas['R_hat'] = alpha_summary['r_hat'].values

# 5. Calculate Half-Life (The "Executive" Metric)
# Formula: ln(0.5) / ln(alpha)
df_alphas['Half_Life_Weeks'] = np.log(0.5) / np.log(df_alphas['Posterior_Alpha'])

# 6. Calculate "90% Dissipation" (How long until the effect is basically gone)
# Formula: ln(0.1) / ln(alpha)
df_alphas['90_Pct_Gone_Weeks'] = np.log(0.1) / np.log(df_alphas['Posterior_Alpha'])

# Format and Display
pd.options.display.float_format = '{:,.3f}'.format
print("--- ALL CHANNELS: ADSTOCK (ALPHA) ANALYSIS ---")
print(df_alphas)

In [ ]:
import arviz as az
import pandas as pd

# 1. Extract the posterior for control coefficients (gamma_c)
summary_gamma = az.summary(mmm.inference_data, var_names=["gamma_c"])

# 2. Get the control names from the inference_data coordinates (The Fail-Safe way)
# Meridian stores coordinate names for controls here:
control_names = mmm.inference_data.posterior.coords['control_variable'].values

# 3. Map the names to the index
summary_gamma.index = control_names

print("--- CONTROL COEFFICIENTS (GAMMA) ---")
# Focus on the mean and the Credible Interval (HDI)
print(summary_gamma[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])

# The "Prior-Posterior Overlap" Test

In [ ]:
import matplotlib.pyplot as plt
import arviz as az
import math
import numpy as np

# 1. Get the data
posterior = mmm.inference_data.posterior
prior = mmm.inference_data.prior

# 2. UPDATED Channel List (Adding SEO to the end)
paid_names = ['G_Search', 'Facebook', 'Display', 'MS_Ads', 'Other', 'OOH']
all_names = paid_names + ['SEO (Organic)']

# 3. Setup the Grid (2 rows, 4 columns for the 7 plots)
num_total = len(all_names)
cols = 4
rows = math.ceil(num_total / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
axes = axes.flatten()

for i in range(num_total):
    ax = axes[i]

    # LOGIC: Check if we are pulling from paid (contribution_m) or organic (contribution_om)
    if i < len(paid_names):
        # Paid Media
        post_data = posterior['contribution_m'][:, :, i].values.flatten()
        prior_data = prior['contribution_m'][:, :, i].values.flatten()
        label_name = paid_names[i]
    else:
        # Organic SEO (contribution_om)
        # Note: Index 0 assumes you have one organic channel ('organic_sessions')
        post_data = posterior['contribution_om'][:, :, 0].values.flatten()
        prior_data = prior['contribution_om'][:, :, 0].values.flatten()
        label_name = "SEO (Organic)"

    # Plotting distributions
    az.plot_dist(prior_data, ax=ax, color="orange", label="Prior (Belief)", kind="kde", fill_kwargs={'alpha': 0.1})
    az.plot_dist(post_data, ax=ax, color="blue", label="Posterior (Data)", kind="kde", fill_kwargs={'alpha': 0.2})

    # Add vertical line for the mean of the posterior
    ax.axvline(np.mean(post_data), color='blue', linestyle='--', alpha=0.6)

    ax.set_title(f"{label_name}", fontweight='bold', fontsize=14)
    ax.set_xlabel("Contribution Share")

    if i == 0:
        ax.legend()

# 4. Clean up unused subplots
for j in range(num_total, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


# New format of outputs export

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generate_meridian_report(mmm, data):
    # 1. Extract Posterior Means
    posterior = mmm.inference_data.posterior

    # Adstock (Alpha)
    alpha_m_mean = posterior['alpha_m'].mean(dim=['chain', 'draw']).values

    # Controls (Gamma)
    gamma_c_mean = posterior['gamma_c'].mean(dim=['chain', 'draw']).values

    # --- CORRECTED NAME RETRIEVAL ---
    # Meridian stores names in the 'coords' of the tensors
    media_names = data.media.coords['media_channel'].values
    control_names = data.controls.coords['control_variable'].values

    # Organic Media handling
    organic_names = []
    alpha_om_mean = []

    if hasattr(data, 'organic_media') and data.organic_media is not None:
        organic_names = data.organic_media.coords['organic_media_channel'].values
        if 'alpha_om' in posterior:
            alpha_om_mean = posterior['alpha_om'].mean(dim=['chain', 'draw']).values
        else:
            alpha_om_mean = [np.nan] * len(organic_names)

    # 2. Calculate Half-Life (Weeks)
    half_life_m = [-np.log(2) / np.log(a) if a > 0 else 0 for a in alpha_m_mean]
    half_life_om = [-np.log(2) / np.log(a) if a > 0 else 0 for a in alpha_om_mean]

    # 3. Print the report
    print("### Copy-Paste the Following Output ###\n")
    print("#### **1. Model Fit & Validity**")
    # Using Meridian's built-in summary if available, otherwise placeholders
    print(f"* **$R^2$ (National):** [Check mmm.summary_metrics for exact value]")
    print(f"* **wMAPE:** [Check mmm.summary_metrics for exact value]")
    print("* **Baseline Status:** [Check plot below]")

    print("\n#### **2. Contribution & Adstock Table**")
    print("| Channel | Posterior Alpha | Half-Life (Weeks) |")
    print("| :--- | :--- | :--- |")

    for i, name in enumerate(media_names):
        print(f"| **{name}** | {alpha_m_mean[i]:.3f} | {half_life_m[i]:.3f} |")

    for i, name in enumerate(organic_names):
        alpha_val = alpha_om_mean[i]
        hl_val = half_life_om[i]
        alpha_str = f"{alpha_val:.3f}" if not np.isnan(alpha_val) else "N/A"
        hl_str = f"{hl_val:.3f}" if not np.isnan(hl_val) else "N/A"
        print(f"| **{name} (Organic)** | {alpha_str} | {hl_str} |")

    print("\n#### **3. Control Coefficients (Gamma)**")
    for i, name in enumerate(control_names):
        if 'ramadan' in name.lower() or 'holiday' in name.lower():
            print(f"* **{name}:** {gamma_c_mean[i]:.3f}")

    print("* **Other significant controls:**")
    for i, name in enumerate(control_names):
        if 'ramadan' not in name.lower() and 'holiday' not in name.lower():
            if abs(gamma_c_mean[i]) > 0.01:
                print(f"    * **{name}:** {gamma_c_mean[i]:.3f}")

    # 4. Baseline Visual Check
    try:
        baseline = posterior['baseline'].mean(dim=['chain', 'draw']).values
        plt.figure(figsize=(12, 4))
        plt.plot(baseline, label='Modeled Baseline', color='black', linewidth=2)
        plt.axhline(0, color='red', linestyle='--', label='Zero Baseline')
        plt.title('Baseline Check')
        plt.grid(alpha=0.3)
        plt.show()
    except:
        pass

# Run it
generate_meridian_report(mmm, mmm.input_data)

#Hill saturation parameters (ec and slope)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def extract_and_plot_saturation(mmm):
    # 1. Extract Posterior Means for Hill Parameters
    posterior = mmm.inference_data.posterior
    data = mmm.input_data

    # Extract ec and slope (handling naming variations in different Meridian versions)
    try:
        ec_m_mean = posterior['ec_m'].mean(dim=['chain', 'draw']).values
        slope_m_mean = posterior['slope_m'].mean(dim=['chain', 'draw']).values
    except KeyError:
        ec_m_mean = posterior['half_max_ec_m'].mean(dim=['chain', 'draw']).values
        slope_m_mean = posterior['slope_m'].mean(dim=['chain', 'draw']).values

    # THE FIX: Accessing names via coordinates
    media_names = data.media.coords['media_channel'].values

    # 2. Print Markdown Table
    print("### Copy-Paste the Following Output ###\n")
    print("#### **4. Saturation (Hill) Parameters**")
    print("| Channel | EC (Half-Max) | Slope |")
    print("| :--- | :--- | :--- |")

    for i, name in enumerate(media_names):
        print(f"| **{name}** | {ec_m_mean[i]:.4f} | {slope_m_mean[i]:.4f} |")

    print("\n---------------------------------------------------")

    # 3. Plot Saturation Curves
    print("\nGenerating Saturation Curves...")

    # Create a synthetic spend range for visualization
    # We use a 0-2 range because Meridian spend is often scaled/normalized
    x_spend = np.linspace(0.01, 2.0, 500)

    plt.figure(figsize=(10, 6))

    for i, name in enumerate(media_names):
        ec = ec_m_mean[i]
        slope = slope_m_mean[i]

        # Hill Function: Saturation = 1 / (1 + (ec / x)**slope)
        y_saturation = 1 / (1 + (ec / x_spend)**slope)

        plt.plot(x_spend, y_saturation, label=f"{name}", linewidth=2)
        # Mark the EC (50% point)
        plt.scatter(ec, 0.5, s=30)

    plt.title('Hill Saturation Curves')
    plt.xlabel('Normalized Spend')
    plt.ylabel('Saturation Level (0-1)')
    plt.axhline(0.5, color='red', linestyle='--', alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# Run the fixed extraction
extract_and_plot_saturation(mmm)

In [ ]:
import pandas as pd
import numpy as np

def extract_march_contributions(mmm):
    # 1. Dynamically find the dimension that represents time
    # We look for names like 'time' or 'date', or the one that isn't geo/media
    all_dims = list(mmm.input_data.media.dims)
    time_dim_name = next((d for d in all_dims if d in ['time', 'date', 'period']), None)

    if time_dim_name is None:
        # Fallback: find the dim that isn't 'geo' or 'media_channel'
        time_dim_name = next((d for d in all_dims if d not in ['geo', 'media_channel']), None)

    if time_dim_name is None:
        print(f"Error: Could not identify time dimension. Found: {all_dims}")
        return

    try:
        raw_dates = mmm.input_data.media.coords[time_dim_name].values
        dates = pd.to_datetime(raw_dates)
    except Exception as e:
        print(f"Error parsing dates from dimension '{time_dim_name}': {e}")
        return

    # 2. Identify March indices
    march_mask = (dates.month == 3)
    march_indices = np.where(march_mask)[0]

    if len(march_indices) == 0:
        print(f"Error: No March dates found in the '{time_dim_name}' dimension.")
        return

    # 3. Locate the Contribution Tensor
    posterior = mmm.inference_data.posterior
    # Common Meridian naming conventions for contribution
    possible_tensors = ['media_contribution', 'incremental_outcome', 'expected_media_contribution']
    tensor_name = next((name for name in possible_tensors if name in posterior), None)

    if tensor_name is None:
        print("Error: Contribution tensor not found. Ensure the model has finished sampling.")
        return

    contribution_tensor = posterior[tensor_name]

    # 4. Slice for March
    # This handles the slice correctly even if time is the 1st or 2nd dimension
    march_contributions = contribution_tensor.isel({time_dim_name: march_indices})

    # 5. Aggregate: Sum over Time, Mean over Chains/Draws, Sum over Geos
    # We sum 'geo' because this is a national report
    geo_dim_name = 'geo' if 'geo' in march_contributions.dims else march_contributions.dims[2]

    march_sum_posterior = march_contributions.sum(dim=[time_dim_name, geo_dim_name])
    march_mean_contributions = march_sum_posterior.mean(dim=['chain', 'draw']).values

    # 6. Final Output
    media_names = mmm.input_data.media.coords['media_channel'].values
    total_march_media_vol = np.sum(march_mean_contributions)

    print(f"--- Results for {tensor_name} ---")
    print(f"Total Modeled Media Volume for March: {total_march_media_vol:,.2f}\n")
    print("| Channel | March Modeled Volume | March Share % |")
    print("| :--- | :--- | :--- |")

    for i, name in enumerate(media_names):
        volume = march_mean_contributions[i]
        share = (volume / total_march_media_vol) * 100 if total_march_media_vol > 0 else 0
        print(f"| **{name}** | {volume:,.2f} | {share:.2f}% |")

# Execute
extract_march_contributions(mmm)

In [ ]:
from meridian.analysis import budget_optimizer

# 1. Define the Total Budget for the period (e.g., the remainder of 2026)
TOTAL_BUDGET = 6963500

# 2. Set Spend Constraints (Min/Max)
# It's usually wise to stay within +/- 30-50% of historical spend
# so the model doesn't suggest unrealistic pivots.
# Order matches: [G_Search, Facebook, Display_Video, MS Ads, The_Other, OOH]
lower_bounds = np.array([0.5, 0.5, 0.3, 0.3, 0.1, 0.8]) # 80% floor for OOH since it's launching
upper_bounds = np.array([1.5, 2.0, 1.5, 2.0, 1.5, 2.0]) # Allowing FB and MS Ads to scale up to 2x

# 3. Initialize and Run the Optimizer
optimizer = budget_optimizer.BudgetOptimizer(
    mmm=mmm,
    total_budget=TOTAL_BUDGET,
    # You can optimize for a specific time window if needed
    # target_time_indices=range(start_idx, end_idx),
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds
)

# This executes the optimization across the posterior draws
optimization_results = optimizer.optimize()

# 4. Extract the Recommended Allocation
recommended_spend = optimization_results.optimal_spend.mean(dim=['chain', 'draw']).values
current_spend = mmm.input_data.media.sum(dim=['time', 'geo']).values

print("### OPTIMIZATION STRATEGY ###\n")
print("| Channel | Current Avg Spend | Recommended Spend | % Change |")
print("| :--- | :--- | :--- | :--- |")

media_names = mmm.input_data.media.coords['media_channel'].values
for i, name in enumerate(media_names):
    change = ((recommended_spend[i] - current_spend[i]) / current_spend[i]) * 100
    print(f"| **{name}** | ${current_spend[i]:,.0f} | ${recommended_spend[i]:,.0f} | {change:+.1f}% |")

# Budget shares


In [ ]:
#Primary channels

import pandas as pd

# 1. Define your main media columns
core_budget_cols = [
    'budget_G Ads',
    'budget_Facebook',
    'budget_Display',
    'budget_MS Ads',
    'budget_The_Other',
    'budget_OOH'
]

# 2. Calculate the "Other" spend
# We assume 'budget_total' is your column for the total weekly/daily spend
total_wallet = df['budget_total'].sum()
core_spend_sum = df[core_budget_cols].sum().sum()
other_spend = total_wallet - core_spend_sum

# 3. Create a summary dataframe
summary_data = df[core_budget_cols].sum().reset_index()
summary_data.columns = ['Channel', 'Total_Spend']

# 4. Append the 'Other' row
other_row = pd.DataFrame([{'Channel': 'other', 'Total_Spend': other_spend}])
budget_df = pd.concat([summary_data, other_row], ignore_index=True)

# 5. Calculate Share %
budget_df['Spend_Share'] = budget_df['Total_Spend'] / total_wallet

# 6. Format for the Slide
print_df = budget_df.copy()
print_df['Spend_Share'] = print_df['Spend_Share'].map('{:.2%}'.format)
print_df['Total_Spend'] = print_df['Total_Spend'].map('{:,.0f}'.format)

print(f"--- Total Marketing Wallet: {total_wallet:,.0f} ---")
print(print_df)

# The Saturation Conclusion
The most important conclusion for OOH is: "Should we spend more?" If your OOH response curve is a straight line, you can scale. If it's flat (concave), you are wasting money.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
filepath='/content/drive/MyDrive'
start_date = '2024-01-15'
end_date = '2026-01-26'
mmm_summarizer.output_model_results_summary('as_summary_output_regroup_imps_submits_adstock_seo_msads_final24feb.html', filepath, start_date, end_date)

Here is a preview of the two-page output based on the simulated data:

In [ ]:
IPython.display.HTML(filename='/content/drive/MyDrive/as_summary_output_regroup_imps_submits_adstock_seo_msads_final24feb.html')

For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>
## Step 5: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [ ]:
filepath = '/content/drive/MyDrive'
optimization_results.output_optimization_summary('as_optimization_output_regroup_imps_submits_adstock_seo_msads_10apr_applications.html', filepath)

In [ ]:
IPython.display.HTML(filename='/content/drive/MyDrive/as_optimization_output_regroup_imps_submits_adstock_seo_msads_10apr_applications.html')

For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).

<a name="save-model"></a>
## Step 6: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [ ]:
file_path='/content/drive/MyDrive/as_mmm_2y_regroup_imps_submits_adstock_seo_msads_10apr_applications.pkl'
model.save_mmm(mmm, file_path)

Run the following codes to load the saved model:

In [ ]:
mmm = model.load_mmm(file_path)